# Random Forest sobre `data_model_borda30`

En esta notebook cargamos el dataset `data_model_borda30`, hacemos una división de datos segura para evitar fuga de datos, optimizamos los hiperparámetros de un modelo Random Forest y evaluamos el modelo usando **Weighted F1 Score** y **AUC-ROC**.

In [1]:
import os
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from sklearn.metrics import classification_report, f1_score, roc_auc_score, accuracy_score, confusion_matrix

random_state = 42
workspace_dir = r'c:\Users\ricar\OneDrive\Desktop\TESIS'
file_path = os.path.join(workspace_dir, 'data_model_borda30.csv')

if not os.path.exists(file_path):
    raise FileNotFoundError(f'No se encontró el archivo: {file_path}')

print('Cargando dataset:', file_path)
data = pd.read_csv(file_path)
print('Shape:', data.shape)
print(data.head())
print('\nColumnas:', data.columns.tolist())
print('\nTipo de label:', data['Type'].dtype)
print('\nDistribución de clases:')
print(data['Type'].value_counts())

Cargando dataset: c:\Users\ricar\OneDrive\Desktop\TESIS\data_model_borda30.csv
Shape: (23779, 31)
   w_t_PEC_A6  w_t_rms_D3  w_f_maxval_D3  w_f_mean_A6      f_std  \
0   12.329105    0.138755     -14.651670   -59.838367  14.393881   
1    3.662787    0.168017     -13.608228   -75.286007  14.257243   
2   21.300689    0.151864     -14.707321   -67.184415  16.163072   
3    3.611111    0.129489     -16.798640   -76.962843  14.356738   
4   17.168839    0.159266     -14.708640   -66.610681  17.097193   

   w_t_peak2peak_D3  t_peak2rms      f_rms     t_rms  w_t_PEC_D3  ...  \
0          1.106960    3.976424  41.141421  0.251482   24.884293  ...   
1          1.316077    3.860386  42.707567  0.259041   39.484031  ...   
2          1.263706    3.848757  41.651699  0.259824   31.675196  ...   
3          0.861959    4.698779  43.799301  0.212821   35.824563  ...   
4          1.186906    3.751130  43.198518  0.266586   33.401840  ...   

   w_f_mean_D1   f_power  w_t_meanEnergyAD  w_f_maxval

In [2]:
label_col = 'Type'
feature_cols = [c for c in data.columns if c != label_col]
X = data[feature_cols].copy()
y = data[label_col].copy()

if X.isna().any().any():
    X = X.fillna(X.median())
    print('Se imputaron valores nulos con la mediana.')

le = LabelEncoder()
y_encoded = le.fit_transform(y)
print('Clases codificadas:', list(le.classes_))

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded,
    test_size=0.2,
    stratify=y_encoded,
    random_state=random_state
)

print('\nTamaño de split:')
print('X_train:', X_train.shape)
print('X_test:', X_test.shape)
print('y_train clase counts:\n', pd.Series(y_train).value_counts(normalize=True))
print('y_test clase counts:\n', pd.Series(y_test).value_counts(normalize=True))

Clases codificadas: ['EXPL', 'HB', 'LP', 'TRE', 'TREMI', 'TRESP', 'VLP', 'VT']

Tamaño de split:
X_train: (19023, 30)
X_test: (4756, 30)
y_train clase counts:
 2    0.529727
7    0.372444
4    0.029964
6    0.023025
1    0.022762
3    0.010619
5    0.007780
0    0.003680
Name: proportion, dtype: float64
y_test clase counts:
 2    0.529857
7    0.372582
4    0.029857
6    0.023129
1    0.022708
3    0.010513
5    0.007780
0    0.003574
Name: proportion, dtype: float64


In [ ]:
# Random Forest no necesita escalado estricto, pero usamos pipeline en caso de extender con otros estimadores.
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestClassifier(random_state=random_state, n_jobs=-1, verbose=1))
])

param_distributions = {
    'rf__n_estimators': [100, 200, 400, 800],
    'rf__max_depth': [None, 10, 20, 30, 50],
    'rf__min_samples_split': [2, 5, 10],
    'rf__min_samples_leaf': [1, 2, 4],
    'rf__max_features': ['sqrt', 'log2', 0.5],
    'rf__class_weight': [None, 'balanced']
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)
search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=30,
    scoring='f1_weighted',
    n_jobs=-1,
    cv=cv,
    verbose=3,
    random_state=random_state,
    return_train_score=True
)

search.fit(X_train, y_train)

print('\nMejores hiperparámetros:')
print(search.best_params_)
print('\nMejor score CV (f1_weighted):', search.best_score_)

Fitting 5 folds for each of 30 candidates, totalling 150 fits

Mejores hiperparámetros:
{'rf__n_estimators': 800, 'rf__min_samples_split': 2, 'rf__min_samples_leaf': 4, 'rf__max_features': 0.5, 'rf__max_depth': 50, 'rf__class_weight': 'balanced'}

Mejor score CV (f1_weighted): 0.8505233852975034


In [4]:
best_model = search.best_estimator_

y_pred = best_model.predict(X_test)
y_pred_proba = best_model.predict_proba(X_test)

f1_weighted = f1_score(y_test, y_pred, average='weighted')
accuracy = accuracy_score(y_test, y_pred)
print('Test accuracy:', accuracy)
print('Test weighted F1:', f1_weighted)

n_classes = len(le.classes_)
if n_classes == 2:
    auc_roc = roc_auc_score(y_test, y_pred_proba[:, 1])
else:
    y_test_bin = label_binarize(y_test, classes=np.arange(n_classes))
    auc_roc = roc_auc_score(y_test_bin, y_pred_proba, average='weighted', multi_class='ovr')

print('Test AUC-ROC (weighted):', auc_roc)

print('\nReporte de clasificación:')
print(classification_report(y_test, y_pred, target_names=le.classes_))

Test accuracy: 0.8456686291000841
Test weighted F1: 0.8426153735308255
Test AUC-ROC (weighted): 0.9505210098508527

Reporte de clasificación:
              precision    recall  f1-score   support

        EXPL       1.00      0.24      0.38        17
          HB       0.56      0.49      0.52       108
          LP       0.87      0.90      0.88      2520
         TRE       0.46      0.24      0.32        50
       TREMI       0.79      0.65      0.72       142
       TRESP       0.63      0.51      0.57        37
         VLP       0.69      0.87      0.77       110
          VT       0.85      0.84      0.84      1772

    accuracy                           0.85      4756
   macro avg       0.73      0.59      0.62      4756
weighted avg       0.84      0.85      0.84      4756



In [5]:
cm = confusion_matrix(y_test, y_pred)
print('Matriz de confusión:')
print(cm)

Matriz de confusión:
[[   4    0    8    0    0    0    0    5]
 [   0   53   21    1    3    0   14   16]
 [   0   12 2264    1    1    0   16  226]
 [   0    4   10   12   12    0    2   10]
 [   0    1   20   10   93   11    3    4]
 [   0    0   10    1    7   19    0    0]
 [   0    2    6    1    2    0   96    3]
 [   0   23  260    0    0    0    8 1481]]


In [6]:
# Guardar modelo final entrenado y resultados adicionales
import joblib

model_path = os.path.join(workspace_dir, 'random_forest_borda30_best.pkl')
joblib.dump(best_model, model_path)
print('Modelo guardado en:', model_path)

results_path = os.path.join(workspace_dir, 'random_forest_borda30_results.txt')
with open(results_path, 'w', encoding='utf-8') as f:
    f.write('Best params:\n')
    f.write(str(search.best_params_) + '\n\n')
    f.write(f'CV f1_weighted: {search.best_score_}\n')
    f.write(f'Test accuracy: {accuracy}\n')
    f.write(f'Test weighted F1: {f1_weighted}\n')
    f.write(f'Test AUC-ROC: {auc_roc}\n\n')
    f.write('Classification report:\n')
    f.write(classification_report(y_test, y_pred, target_names=le.classes_))
    f.write('\nConfusion matrix:\n')
    f.write(np.array2string(cm))

print('Resultados guardados en:', results_path)

Modelo guardado en: c:\Users\ricar\OneDrive\Desktop\TESIS\random_forest_borda30_best.pkl
Resultados guardados en: c:\Users\ricar\OneDrive\Desktop\TESIS\random_forest_borda30_results.txt
